# Lecture 21: Multi-Armed Bandits in Persuasion

**Learning Objectives:**
1. Understand the explore/exploit tradeoff
2. Compare three bandit algorithms: epsilon-greedy, UCB, Thompson sampling
3. See how bandits work in political messaging (Offer-Westort) and marketing (Schwartz-Bradlow-Fader)
4. Understand when learning happens vs. when it fails (Meyer & Hutchinson, Salganik)

**Papers:**
- Offer-Westort, Coppock & Green (2021): Thompson sampling for political message testing
- Schwartz, Bradlow & Fader (2017): Thompson sampling for customer acquisition (750M impressions)
- Meyer & Hutchinson (2016): When do people learn to balance explore/exploit?


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
print('✓ Ready to run bandits simulations')

## The Setup: 5 Message Variants

You're running a political campaign and have 5 different message framings:

1. **Economic message**: Focus on job creation
2. **Healthcare message**: Focus on affordability
3. **Education message**: Focus on opportunity
4. **Climate message**: Focus on future
5. **Unity message**: Focus on bringing people together

Each message has a true persuasion rate (unknown to you). Your goal: learn which is best, but ALSO show the best ones more often.


In [ ]:
# True persuasion rates (you don't know these during the campaign)
true_rates = np.array([0.10, 0.12, 0.08, 0.07, 0.11])  # Economic, Healthcare, Education, Climate, Unity
message_names = ['Economic', 'Healthcare', 'Education', 'Climate', 'Unity']

print('True persuasion rates (secret):')\nfor i, (name, rate) in enumerate(zip(message_names, true_rates)):
    print(f'  {i+1}. {name}: {rate:.1%}')
print(f'\nBest message: {message_names[np.argmax(true_rates)]} ({np.max(true_rates):.1%})')

## Algorithm 1: Epsilon-Greedy

Strategy: 90% of the time, show the best message so far. 10% of the time, try a random message.


In [ ]:
def epsilon_greedy(true_rates, n_rounds=2000, epsilon=0.1):
    """Epsilon-greedy bandit."""
    n_arms = len(true_rates)
    clicks = np.zeros(n_arms)
    impressions = np.zeros(n_arms)
    allocation = []
    rewards = []
    
    for t in range(n_rounds):
        # Decide which message to show
        if np.random.rand() < epsilon:  # Explore
            arm = np.random.randint(n_arms)
        else:  # Exploit
            obs_rates = clicks / np.maximum(impressions, 1)
            arm = np.argmax(obs_rates)
        
        # Show the message, get reward
        impressions[arm] += 1
        reward = int(np.random.rand() < true_rates[arm])
        clicks[arm] += reward
        allocation.append(arm)
        rewards.append(reward)
    
    return clicks, impressions, allocation, rewards

clicks_eg, impr_eg, alloc_eg, rewards_eg = epsilon_greedy(true_rates)

print('\n=== EPSILON-GREEDY RESULTS ===')
for i, name in enumerate(message_names):
    obs_rate = clicks_eg[i] / impr_eg[i] if impr_eg[i] > 0 else 0
    print(f'{name:12} | Impressions: {int(impr_eg[i]):4} | Clicks: {int(clicks_eg[i]):3} | Rate: {obs_rate:.1%}')

print(f'\nTotal persuaded: {int(np.sum(clicks_eg))}')

## Algorithm 2: Upper Confidence Bound (UCB)

Strategy: For each message, compute observed rate + uncertainty bonus. Show the message with the highest total.


In [ ]:
def ucb_bandit(true_rates, n_rounds=2000, c=1.0):
    """UCB bandit."""
    n_arms = len(true_rates)
    clicks = np.zeros(n_arms)
    impressions = np.zeros(n_arms)
    allocation = []
    rewards = []
    
    for t in range(n_rounds):
        # Compute UCB for each arm
        obs_rates = clicks / np.maximum(impressions, 1)
        uncertainty = c * np.sqrt(np.log(t + 2) / np.maximum(impressions, 1))
        ucb_values = obs_rates + uncertainty
        
        # Show message with highest UCB
        arm = np.argmax(ucb_values)
        
        # Get reward
        impressions[arm] += 1
        reward = int(np.random.rand() < true_rates[arm])
        clicks[arm] += reward
        allocation.append(arm)
        rewards.append(reward)
    
    return clicks, impressions, allocation, rewards

clicks_ucb, impr_ucb, alloc_ucb, rewards_ucb = ucb_bandit(true_rates)

print('\n=== UCB RESULTS ===')
for i, name in enumerate(message_names):
    obs_rate = clicks_ucb[i] / impr_ucb[i] if impr_ucb[i] > 0 else 0
    print(f'{name:12} | Impressions: {int(impr_ucb[i]):4} | Clicks: {int(clicks_ucb[i]):3} | Rate: {obs_rate:.1%}')

print(f'\nTotal persuaded: {int(np.sum(clicks_ucb))}')

## Algorithm 3: Thompson Sampling

Strategy (Offer-Westort, Schwartz-Bradlow-Fader): For each message, maintain a belief distribution. Sample from each, show the one with the highest sample.


In [ ]:
def thompson_sampling(true_rates, n_rounds=2000):
    """Thompson sampling with Beta-Bernoulli model."""
    n_arms = len(true_rates)
    successes = np.ones(n_arms)  # Beta prior: successes (α)
    failures = np.ones(n_arms)   # Beta prior: failures (β)
    allocation = []
    rewards = []
    
    for t in range(n_rounds):
        # Sample from each arm's Beta distribution
        samples = np.random.beta(successes, failures)
        arm = np.argmax(samples)
        
        # Get reward
        reward = int(np.random.rand() < true_rates[arm])
        if reward:
            successes[arm] += 1
        else:
            failures[arm] += 1
        
        allocation.append(arm)
        rewards.append(reward)
    
    clicks = successes - 1  # Remove the prior
    impressions = successes + failures - 2
    return clicks, impressions, allocation, rewards

clicks_ts, impr_ts, alloc_ts, rewards_ts = thompson_sampling(true_rates)

print('\n=== THOMPSON SAMPLING RESULTS ===')
for i, name in enumerate(message_names):
    obs_rate = clicks_ts[i] / impr_ts[i] if impr_ts[i] > 0 else 0
    print(f'{name:12} | Impressions: {int(impr_ts[i]):4} | Clicks: {int(clicks_ts[i]):3} | Rate: {obs_rate:.1%}')

print(f'\nTotal persuaded: {int(np.sum(clicks_ts))}')

## Comparison: Which algorithm wins?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Epsilon-Greedy
alloc_counts_eg = np.bincount(alloc_eg, minlength=5)
axes[0].bar(message_names, alloc_counts_eg, color='steelblue')
axes[0].set_title('Epsilon-Greedy\nAllocation')
axes[0].set_ylabel('Impressions')
axes[0].tick_params(axis='x', rotation=45)

# UCB
alloc_counts_ucb = np.bincount(alloc_ucb, minlength=5)
axes[1].bar(message_names, alloc_counts_ucb, color='darkgreen')
axes[1].set_title('UCB\nAllocation')
axes[1].set_ylabel('Impressions')
axes[1].tick_params(axis='x', rotation=45)

# Thompson Sampling
alloc_counts_ts = np.bincount(alloc_ts, minlength=5)
axes[2].bar(message_names, alloc_counts_ts, color='darkred')
axes[2].set_title('Thompson Sampling\nAllocation')
axes[2].set_ylabel('Impressions')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Total reward comparison
print('\n=== TOTAL PERFORMANCE ===')
print(f'Epsilon-Greedy:    {int(np.sum(rewards_eg))} persuaded')
print(f'UCB:               {int(np.sum(rewards_ucb))} persuaded')
print(f'Thompson Sampling: {int(np.sum(rewards_ts))} persuaded')
print(f'\nIf you showed only the best message: {int(2000 * np.max(true_rates))} persuaded')

## Reflection Questions (answer in the Google Form)

1. **Which algorithm persuaded the most people?** Why do you think that is?

2. **Offer-Westort found that Thompson Sampling identified the winning message faster than fixed A/B testing.** In this simulation, how many impressions did Thompson Sampling need to correctly identify the Healthcare message (the true best) with high confidence?

3. **Meyer & Hutchinson studied when people learn to balance exploration and exploitation.** If voters were "learning" which message works best through their own exposure (rather than an algorithm deciding), what would change? Would they do as well as Thompson Sampling?

4. **Schwartz, Bradlow & Fader ran their bandit on 750 million impressions and got an 8% lift.** In this simulation, what was the lift from using Thompson Sampling instead of random allocation (showing each message equally)?
